# QuantOne v2 — Phase 3 dry run (qwen35_4b, all quants)

Runs ONE model through every quant with the thorough-v2 knobs, scores it, and checks the ⛔ gates before any bulk spend:
- **G1** parse rate at FP16 > 0.50 (else chat-template bug → stop)
- **G2** decline-subset CIs now ≤ ±0.08 (the n=30→200 power fix)
- **G3** zero think-leaks (the /no_think enforcement holds on the model that broke in Phase 0)

GPU T4 required. Resume ritual identical to v1: Save Version → attach output as input → re-run until the manifest is complete, then the gate cell runs.

In [ ]:
MODEL = "qwen35_4b"
REPO_GIT_URL = "https://github.com/kaushiksai29/QuantOne.git"
BRANCH = "v2"

In [ ]:
import os, shutil, subprocess
def gpu_name():
    if shutil.which('nvidia-smi') is None: return None
    r = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode==0 and r.stdout.strip() else None
if not os.path.exists('QuantOne'):
    !git clone -b {BRANCH} {REPO_GIT_URL}
%cd QuantOne
GPU = gpu_name()
if GPU is None:
    raise SystemExit('STOP: no GPU. Enable T4 accelerator, then Run All.')
print('GPU:', GPU)
!pip install -q llama-cpp-python==0.3.33 pandas pyarrow jsonschema PyYAML --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
try:
    import llama_cpp
except Exception as exc:
    print(f'building from source ({exc}) ...')
    !pip uninstall -y -q llama-cpp-python
    os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
    !pip install -q llama-cpp-python==0.3.33 --no-binary llama-cpp-python
    import llama_cpp
print('llama-cpp-python', llama_cpp.__version__)

In [ ]:
# Ingest any prior dry-run results + run all quants for this model
import glob
prior = glob.glob('/kaggle/input/*/results_v2_*.zip')
prior_args = ' '.join(f'--prior-results {p}' for p in prior)
print('prior zips:', prior or 'none')
!python scripts/session_v2.py --model {MODEL} {prior_args} \
    --n-gpu-layers -1 --results-dir /kaggle/working/results_v2 \
    --zip-to /kaggle/working/results_v2_{MODEL}.zip

In [ ]:
# Score + gate. If the manifest above is complete this prints the verdict.
!python scorer/score_v2.py --results-dir /kaggle/working/results_v2 --out /kaggle/working/scores_v2.parquet
!python scripts/dryrun_gate.py --scores /kaggle/working/scores_v2.parquet --model {MODEL}